<a href="https://colab.research.google.com/github/srki-cs/QUANTISCAN/blob/main/QUANTISCAN_P1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =============================================================================
# ENVIRONMENT SETUP & FALLBACK
# =============================================================================
import os

try:
    from google.colab import drive
    print("Attempting to connect to Google Drive...")
    drive.mount('/content/drive', timeout_ms=5000)
    USE_DRIVE = True
except Exception as e:
    print("Google Drive connection skipped or unavailable. Running in Local Mode.")
    USE_DRIVE = False

Mounted at /content/drive


In [ ]:
# ==============================================================================
# LIBRARIES
# ==============================================================================

import numpy as np
import pandas as pd
from skimage.data import shepp_logan_phantom
from skimage.transform import resize, radon, iradon
from skimage.metrics import structural_similarity as ssim
import warnings, itertools, os
from itertools import combinations

# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================

# DICTIONARY
PHANTOM_SIZE      = 256
ANGLE_LEVELS      = [10, 20, 40, 60, 90, 180]
PHOTON_LEVELS     = [1e2, 1e3, 1e4, 1e5]
N_TRIALS          = 10
MUL_MAX           = 5.2

#LESION
LESION_CENTER   = (90, 128)
LESION_RADIUS   = 8
BG_INNER_R      = 14
BG_OUTER_R      = 26
LESION_CONTRAST = 0.15
#===============================================================================
if USE_DRIVE and os.path.exists("/content/drive"):
    OUTPUT_DIR = "/content/drive/MyDrive/Experiment/QUANTISCAN/outputs"
else:
    OUTPUT_DIR = "./outputs"

print(f"Active Output Directory: {OUTPUT_DIR}")
# ==============================================================================
# 2. PHANTOM CONSTRUCTION
# ==============================================================================


def build_phantom():
  ph = shepp_logan_phantom()
  ph = resize(ph, (PHANTOM_SIZE, PHANTOM_SIZE), anti_aliasing=True)
  ph = (ph - ph.min())/(ph.max()-ph.min()) #Normalization Process for which the max pix value = 1 and the lowest = 0
  rr, cc = np.ogrid[:PHANTOM_SIZE, :PHANTOM_SIZE] #This line is to create an array for column and rows across the whole image
  mask = (rr - LESION_CENTER[0])**2 + (cc - LESION_CENTER[1])**2 <= LESION_RADIUS**2 #LESION EMBBED
  pre_lesion_mean = ph[mask].mean() #Checks whether the lesion is properly embedded
  if pre_lesion_mean > 0.6:
    raise ValueError(
        f"Lesion was placed in high-intensity region (mean = {pre_lesion_mean:.3f})"
        "Please udjust -LESION_CENTER-"
    )
  ph[mask] = np.clip(ph[mask] + LESION_CONTRAST, 0, 1)
  print(f" Lesion was sucessfully inserted at {LESION_CENTER},"
        f" background mean: {pre_lesion_mean: .3f}, contrast : +{LESION_CONTRAST}")
  return ph

# ==============================================================================
# 3. FORWARD TRANSFORM
# ==============================================================================

def forward_transform(phantom, n_angles):
  angles = np.linspace(0, 180, n_angles, endpoint=False)
  sinogram = radon(phantom, theta=angles, circle=True)
  return sinogram, angles


# ==============================================================================
# 4. POISSON NOISE MODELING
# ==============================================================================

def add_poisson_noise(sinogram, I0):
  sino_norm     = sinogram / (sinogram.max() + 1e-10) #normalization | adding 1e-10 prevents error by dividing zero later on
  muL           = MUL_MAX * sino_norm
  transmitted   = I0 * np.exp(-muL)
  noisy_I       = np.random.poisson(np.maximum(transmitted, 1e-6).astype(float))
  noisy_I       = np.maximum(noisy_I, 1)
  noisy_muL     = -np.log(noisy_I / I0)
  noisy_sino    = (noisy_muL / MUL_MAX) * (sinogram.max() + 1e-10)
  return noisy_sino

# ==============================================================================
# 5. FILTERED BACK PROJECTION RECONSTRUCTION
# ==============================================================================
def fbp_reconstruction(sinogram, angles):
  recon = iradon(sinogram, theta=angles, filter_name='ramp', circle= True)
  recon = np.clip(recon, 0, None)
  if recon.max() > 0:
    recon /= recon.max()
  return recon

# ==============================================================================
# 6. QUALITY METRICS
# ==============================================================================
# 6.a CNR
# ==============================================================================
def compute_cnr(reconstructed):
  rr, cc       = np.ogrid[:PHANTOM_SIZE, :PHANTOM_SIZE]
  dist         = np.sqrt((rr - LESION_CENTER[0])**2 + (cc - LESION_CENTER[1])**2)
  lesion_mask  = dist <= LESION_RADIUS
  bg_mask      = (dist > BG_INNER_R) & (dist <= BG_OUTER_R)
  lesion_vals  = reconstructed[lesion_mask]
  bg_vals      = reconstructed[bg_mask]
  sd_bg        = bg_vals.std(ddof=1)
  if sd_bg < 1e-10:
    return 0.0
  return abs((lesion_vals.mean()-bg_vals.mean())/sd_bg)
# ==============================================================================
# 6.b PSNR

def compute_psnr(reference, reconstructed):
  mse = np.mean((reference - reconstructed) ** 2)
  if mse < 1e-12:
    return 100.0
  return 10 * np.log10(1.0 / mse)

## ==============================================================================
# 6.b SSIM

def compute_ssim(reference, reconstructed):
  return ssim(reference, reconstructed, data_range=1.0)

# ==============================================================================
# 7. MAIN SIMULATION LOOP
# ==============================================================================

def run_simulation(phantom):
  records = []
  print("─" * 65)
  for n_angles, I0 in itertools.product(ANGLE_LEVELS, PHOTON_LEVELS):
    sinogram_clean, angles = forward_transform(phantom, n_angles)
    for trial in range(1, N_TRIALS +1):
      np.random.seed(trial * 1000 + int(I0))
      noisy_sinogram = add_poisson_noise(sinogram_clean, I0)
      recon          = fbp_reconstruction(noisy_sinogram, angles)
      cnr            = compute_cnr(recon)
      psnr           = compute_psnr(phantom, recon)
      ssim           = compute_ssim(phantom, recon)
      records.append({'angles': n_angles, 'photons': I0,
                'trial': trial, 'ssim': ssim, 'psnr': psnr, 'cnr': cnr})
  return pd.DataFrame(records)

# ==============================================================================
# 8. RUN SIMULATION
# ==============================================================================

def run_all():
  phantom = build_phantom()
  df_result = run_simulation(phantom)
  return df_result

# ==============================================================================
# 9. SAVE CSV FILE
# ==============================================================================
def save_result(df):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    csv_path = os.path.join(OUTPUT_DIR, "P1_SIMULATION_RESULTS.csv")
    df.to_csv(csv_path, index=False)
    print(f"Results were successfully saved to: {csv_path}")








In [ ]:
df_result = run_all()
save_result(df_result)

 Lesion was sucessfully inserted at (90, 128), background mean:  0.298, contrast : +0.15
─────────────────────────────────────────────────────────────────
Results were successfully saved to: /content/drive/MyDrive/Experiment/QUANTISCAN/outputs/P1_SIMULATION_RESULTS.csv
